In [ ]:
# Install required packages (auto-skipped if already installed)
import importlib
if importlib.util.find_spec('qiskit') is None:
    !pip install -q qiskit qiskit-aer qiskit-ibm-runtime pylatexenc numpy
else:
    print("\u2713 Packages already installed")

# To run on real quantum hardware, uncomment and fill in your credentials:
# from qiskit_ibm_runtime import QiskitRuntimeService
# QiskitRuntimeService.save_account(
#     token="<your-api-key>",
#     instance="<your-crn>",
#     overwrite=True
# )

# Especificar observáveis na base de Pauli

<details>
<summary><b>Versões dos pacotes</b></summary>

O código nesta página foi desenvolvido utilizando os seguintes requisitos.
Recomendamos o uso dessas versões ou versões mais recentes.

```
qiskit[all]~=2.3.0
```
</details>

Na mecânica quântica, os observáveis correspondem a propriedades físicas que podem ser medidas.
Ao considerar um sistema de spins, por exemplo, você pode estar interessado em medir a energia do sistema ou obter informações sobre o alinhamento dos spins, como a magnetização ou as correlações entre spins.

Para medir um observável $O$ de $n$ qubits em um computador quântico, você deve representá-lo como uma soma de produtos tensoriais de operadores de Pauli, ou seja

$$
O = \sum_{k=1}^K \alpha_k P_k,~~ P_k \in {I, X, Y, Z}^{\otimes n},~~ \alpha_k \in \mathbb{R},
$$

onde

$$
I = \begin{pmatrix}
1 & 0 \\ 0 & 1
\end{pmatrix}
~~
X = \begin{pmatrix}
0 & 1 \\ 1 & 0
\end{pmatrix}
~~
Y = \begin{pmatrix}
0 & -i \\ i & 0
\end{pmatrix}
~~
Z = \begin{pmatrix}
1 & 0 \\ 0 & -1
\end{pmatrix}
$$

e você utiliza o fato de que um observável é hermitiano, isto é, $O^\dagger = O$. Se $O$ não for hermitiano, ainda é possível decompô-lo como uma soma de Paulis, mas o coeficiente $\alpha_k$ se torna complexo.

Em muitos casos, o observável é naturalmente especificado nessa representação após mapear o sistema de interesse para qubits.
Por exemplo, um sistema de spin-1/2 pode ser mapeado para um Hamiltoniano de Ising

$$
H = \sum_{\langle i, j\rangle} Z_i Z_j - \sum_{i=1}^n X_i,
$$

onde os índices $\langle i, j\rangle$ percorrem os spins em interação e os spins estão sujeitos a um campo transversal em $X$.
O índice subscrito indica em qual qubit o operador de Pauli atua, ou seja, $X_i$ aplica um operador $X$ no qubit $i$ e deixa os demais inalterados.

No Qiskit SDK, este Hamiltoniano pode ser construído com o seguinte código.

In [1]:
from qiskit.quantum_info import SparsePauliOp

# define the number of qubits
n = 12

# define the single Pauli terms as ("Paulis", [indices], coefficient)
interactions = [
    ("ZZ", [i, i + 1], 1) for i in range(n - 1)
]  # we assume spins on a 1D line
field = [("X", [i], -1) for i in range(n)]

# build the operator
hamiltonian = SparsePauliOp.from_sparse_list(
    interactions + field, num_qubits=n
)
print(hamiltonian)

SparsePauliOp(['IIIIIIIIIIZZ', 'IIIIIIIIIZZI', 'IIIIIIIIZZII', 'IIIIIIIZZIII', 'IIIIIIZZIIII', 'IIIIIZZIIIII', 'IIIIZZIIIIII', 'IIIZZIIIIIII', 'IIZZIIIIIIII', 'IZZIIIIIIIII', 'ZZIIIIIIIIII', 'IIIIIIIIIIIX', 'IIIIIIIIIIXI', 'IIIIIIIIIXII', 'IIIIIIIIXIII', 'IIIIIIIXIIII', 'IIIIIIXIIIII', 'IIIIIXIIIIII', 'IIIIXIIIIIII', 'IIIXIIIIIIII', 'IIXIIIIIIIII', 'IXIIIIIIIIII', 'XIIIIIIIIIII'],
              coeffs=[ 1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,  1.+0.j,
  1.+0.j,  1.+0.j,  1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j,
 -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j, -1.+0.j])


If we would like to measure the energy the observable is the Hamiltonian itself. Alternatively, we could be
interested in measuring system properties like the average magnetization by counting the number of spins
aligned in the $Z$-direction with the observable

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

For observables that are not given in terms of Pauli operators but in a matrix form, we first have to reformulate them
in the Pauli basis in order to evaluate them on a quantum computer.
We are always able to find such a representation as the Pauli matrices form a basis for the Hermitian $2^n \times 2^n$ matrices.
We expand the observable $O$ as

$$
O = \sum_{P \in \{I, X, Y, Z\}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

where the sum runs over all possible $n$-qubit Pauli terms and $\mathrm{Tr}(\cdot)$ is the trace of a matrix, which acts as inner product.
You can implement this decomposition  from a matrix to Pauli terms using the `SparsePauliOp.from_operator` method, like so:

In [2]:
import numpy as np
from qiskit.quantum_info import SparsePauliOp

matrix = np.array(
    [[-1, 0, 0.5, -1], [0, 1, 1, 0.5], [0.5, 1, -1, 0], [-1, 0.5, 0, 1]]
)

observable = SparsePauliOp.from_operator(matrix)
print(observable)

SparsePauliOp(['IZ', 'XI', 'YY'],
              coeffs=[-1. +0.j,  0.5+0.j,  1. -0.j])


Se quisermos medir a energia, o observável é o próprio Hamiltoniano. Alternativamente, podemos estar
interessados em medir propriedades do sistema, como a magnetização média, contando o número de spins
alinhados na direção $Z$ com o observável

$$
O = \frac{1}{n} \sum_{i=1} Z_i
$$

Para observáveis que não são dados em termos de operadores de Pauli, mas na forma de matriz, precisamos primeiro reformulá-los
na base de Pauli para avaliá-los em um computador quântico.
Podemos sempre encontrar tal representação, pois as matrizes de Pauli formam uma base para as matrizes hermitianas $2^n \times 2^n$.
Expandimos o observável $O$ como

$$
O = \sum_{P \in {I, X, Y, Z}^{\otimes n}} \mathrm{Tr}(O P) P,
$$

onde a soma percorre todos os possíveis termos de Pauli de $n$ qubits e $\mathrm{Tr}(\cdot)$ é o traço de uma matriz, que atua como produto interno.
Você pode implementar essa decomposição de uma matriz para termos de Pauli usando o método `SparsePauliOp.from_operator`, da seguinte forma:

In [3]:
from qiskit.circuit import QuantumCircuit

# create a circuit, where we would like to measure
# q0 in the X basis, q1 in the Y basis and q2 in the Z basis
circuit = QuantumCircuit(3)
circuit.ry(0.8, 0)
circuit.cx(0, 1)
circuit.cx(1, 2)
circuit.barrier()

# diagonalize X with the Hadamard gate
circuit.h(0)

# diagonalize Y with Hadamard as S^\dagger
circuit.sdg(1)
circuit.h(1)

# the Z basis is the default, no action required here

# measure all qubits
circuit.measure_all()
circuit.draw("mpl")

<Image src="../docs/images/guides/specify-observables-pauli/extracted-outputs/ce4b1984-ebe0-44f6-a78c-d67b9e9bb361-0.svg" alt="Output of the previous code cell" />

## Next steps

<Admonition type="tip" title="Recommendations">
  -  Read the [SparsePauliOp API](/docs/api/qiskit/qiskit.quantum_info.SparsePauliOp#sparsepauliop) reference.
</Admonition>